In [ ]:
import numpy as np
import torch
import torch.nn as nn

In [4]:
M = 100
N = 1000
T = 1

In [11]:
W = np.zeros(shape=(M+1,N))
X = np.zeros(shape=(M+1,N))
tv = np.linspace(1,2,N)
sv = np.linspace(1,2,N)
for i in range(N):
    for t in range(1,M+1):
        dt = tv[t]-tv[t-1]
        W[t,i] = W[t-1,i]+np.sqrt(dt)*np.random.normal(0,1,1)
        X[t,i] = sv[i]*W[t,i]

dX = X[1:,:]-X[:-1,:]
SVT = torch.Tensor(sv)
dXT = torch.Tensor(dX)

dXT.shape

C:\Users\trist\AppData\Local\Temp\ipykernel_29536\2678332862.py:8: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  W[t,i] = W[t-1,i]+np.sqrt(dt)*np.random.normal(0,1,1)


torch.Size([100, 1000])

In [7]:
Phi = nn.Sequential(
    nn.Linear(M,16),
    nn.ReLU(),
    nn.Linear(16,16),
    nn.ReLU(),
    nn.Linear(16,1)
)

In [8]:
eta = 0.001
epochs = 100
lossFn = nn.MSELoss()
opt = torch.optim.SGD(lr=eta,params=Phi.parameters())
bs = 32

In [12]:
for e in range(1,epochs+1):
    order = torch.randperm(N)
    for b in range(0,bs,N):
        L = 0
        opt.zero_grad()
        for i in order[b:b+bs]:
            bsL = len(order[b:b+bs])
            pred = Phi(dXT[:,i])
            act = SVT[i]
            L += lossFn(pred,act)
        L = L/bsL
        L.backward()
        opt.step()

C:\Users\trist\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\nn\modules\loss.py:538: UserWarning: Using a target size (torch.Size([])) that is different to the input size (torch.Size([1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


In [19]:
print(Phi(dXT[:,100]))
print(SVT[100])

tensor([0.2515], grad_fn=<ViewBackward0>)
tensor(1.1001)
